# Notebook 4 (bonus, advanced) — The multimodal, multiscale picture

**MDIG2026 Summer School · NLP track · bonus**

Notebooks 1–3 stayed inside language. This bonus notebook connects language to the corpus's
*other* channels — the whole point of a **multiscale** summer school. For a single trial we
put three time-locked streams on one shared clock:

| stream | scale | source |
|--------|-------|--------|
| **semantic homing** — clue meaning → target | slow, cognitive | transcripts (Nb 2) |
| **gesture bouts** — the clue-giver's hands | mid, motor | `gestureclassifications/…_segments.csv` |
| **postural sway** — the balance board | fast, bodily | `gyroscope.csv` |

Then we ask a cross-modal question across trials: **is semantic homing accompanied by more
gesturing and more sway?**

> Prereqs: run **Notebook 1** with `MAX_TRIALS = None`, then peek at **Notebook 2**.
> Everything is time-aligned to *trial start = 0 s* (the gyro is cut per trial; gesture times
> and transcript times are already trial-relative).

In [ ]:
# Use locally-cached models only — never phone home to Hugging Face.
# (The embedding model is already downloaded; this prevents a hang if the
#  network blocks HF's version-check request. Must run before any model load.)
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("Hugging Face offline mode ON — using cached models only.")

## 0 · Install (run once)

In [ ]:
# %pip install sentence-transformers pandas numpy matplotlib

## 1 · Config, transcripts, embedding model

In [ ]:
import os, certifi
if not os.path.exists(os.environ.get("SSL_CERT_FILE", "")):   # fix broken Anaconda cert path
    os.environ["SSL_CERT_FILE"] = certifi.where(); os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
def find_corpus(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(8):
        if os.path.isfile(os.path.join(d, "metadata.csv")) and os.path.isdir(os.path.join(d, "audios")):
            return d
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    raise FileNotFoundError("Could not find the BalanceCorpus root (metadata.csv + audios/).")
CORPUS = find_corpus()
TRANSCRIPTS = os.path.join(CORPUS, "scripts", "nlp_taboo", "transcripts")
print("Corpus root:", CORPUS)

In [ ]:
import pandas as pd, numpy as np
tx_path = os.path.join(TRANSCRIPTS, "transcripts_all.csv")
if not os.path.exists(tx_path):
    raise FileNotFoundError("Run Notebook 1 first (MAX_TRIALS = None) to create transcripts_all.csv.")
tx = pd.read_csv(tx_path, encoding="utf-8")
meta = pd.read_csv(os.path.join(CORPUS, "metadata.csv"), encoding="utf-8-sig")
meta["pair_id"] = meta["pair_id"].astype(str)
print(f"{len(tx)} segments · {tx.trial_id.nunique()} trials")

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
except ModuleNotFoundError:
    raise ModuleNotFoundError("sentence-transformers is not installed — run the install cell "
                              "in Section 0, then re-run this cell.")
EMB = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
def embed(texts): return EMB.encode(list(texts), normalize_embeddings=True, show_progress_bar=False)
print("Embedding model ready.")

## 2 · Three loaders — one per stream
Each returns trial-relative time so the streams line up.

In [ ]:
# --- (a) semantic homing: cumulative clue similarity to the target ---
def homing(trial_id):
    d = tx[(tx.trial_id == trial_id) & (tx.role == "clue_giver")].sort_values("start")
    if len(d) == 0: return None
    target = tx[tx.trial_id == trial_id].target_word.iloc[0]
    tvec = embed([target])[0]
    cum, ts, sims = "", [], []
    for _, s in d.iterrows():
        cum = (cum + " " + str(s.text)).strip()
        ts.append(s.end); sims.append(float(embed([cum])[0] @ tvec))
    return np.array(ts), np.array(sims), target

In [ ]:
# --- (b) postural sway: RMS of angular-velocity magnitude from the balance board ---
import glob
GYRO = pd.read_csv(os.path.join(CORPUS, "gyroscope.csv"))
# angular-velocity columns carry a degree symbol that may be mojibaked -> match by prefix
AS = [next(c for c in GYRO.columns if c.startswith(p)) for p in ("AsX", "AsY", "AsZ")]

def sway(pair_id, trial_number, smooth=10):
    g = GYRO[(GYRO.group_name == pair_id) & (GYRO.trial_number == int(trial_number))].copy()
    if len(g) == 0: return None
    t = pd.to_datetime(g["time"]); secs = (t - t.min()).dt.total_seconds().to_numpy()
    mag = np.sqrt((g[AS].to_numpy() ** 2).sum(axis=1))              # deg/s, total rotation rate
    order = np.argsort(secs); secs, mag = secs[order], mag[order]   # gyro rows are NOT time-sorted: sort BEFORE smoothing
    env = pd.Series(mag).rolling(smooth, center=True, min_periods=1).mean().to_numpy()
    return secs, env

In [ ]:
# --- (c) gesture bouts: the clue-giver's hand-gesture segments (CNN model) ---
def gesture_bouts(trial_id):
    stem = trial_id + "_clueGiver_cam02.mp4_segments.csv"
    path = os.path.join(CORPUS, "gestureclassifications", stem)
    if not os.path.exists(path): return []
    s = pd.read_csv(path)
    s = s[s.model == "CNN"] if "model" in s else s
    return list(zip(s.start_time, s.end_time))

## 3 · One trial, three streams, one clock

In [ ]:
import matplotlib.pyplot as plt

# choose a trial that has all three streams
def has_all(t):
    r = tx[tx.trial_id == t].iloc[0]
    return (homing(t) is not None and sway(r.pair_id, r.trial_number) is not None
            and len(gesture_bouts(t)) > 0)
example = next((t for t in tx.trial_id.unique() if has_all(t)), None)
if example is None:
    raise ValueError("No trial currently has all three streams (transcript + gyroscope + gesture). "
                     "Run Notebook 1 on more trials, and check the gyroscope/gesture files are present.")
r = tx[tx.trial_id == example].iloc[0]

H = homing(example); S = sway(r.pair_id, r.trial_number); B = gesture_bouts(example)
ht, hs, target = H; st, se = S

fig, ax = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
ax[0].plot(ht, hs, "-o", color="#c44"); ax[0].set_ylabel(f"cosine to\n'{target}'")
ax[0].set_title(f"Multimodal timeline · {example}  ({r.condition})")
ax[1].plot(st, se, color="#2a7"); ax[1].set_ylabel("board sway\n(deg/s)")
for (a, b) in B:
    for row in ax: row.axvspan(a, b, color="#48a", alpha=.12)
ax[2].set_ylim(0, 1); ax[2].set_yticks([])
for (a, b) in B: ax[2].axvspan(a, b, color="#48a", alpha=.5)
ax[2].set_ylabel("gesture\nbouts"); ax[2].set_xlabel("time in trial (s)")
plt.tight_layout(); plt.show()
print("Blue bands = clue-giver gesturing. Do homing rises coincide with gesture/sway bursts?")

## 4 · Cross-modal, across trials

For every trial we reduce each stream to one number and look for relationships:

- **homing slope** — how fast meaning approaches the target (linear fit over normalized time)
- **gesture fraction** — share of the trial spent gesturing
- **mean sway** — average board rotation rate

> **Expect ground sway ≈ 0.** In *ground* trials there is no balance board to stabilise, so the
> board's IMU barely moves (~0 °/s) while *board* trials average ~8 °/s. That flat ground line is
> the real manipulation working, not a bug.

Exploratory only (2 dyads) — read the *pattern*, colour-coded by board vs ground.

In [ ]:
rec = []
for t in tx.trial_id.unique():
    r = tx[tx.trial_id == t].iloc[0]
    H = homing(t); S = sway(r.pair_id, r.trial_number); B = gesture_bouts(t)
    if H is None or S is None or len(H[0]) < 3: continue
    ht, hs, _ = H
    tn = (ht - ht.min()) / (ht.max() - ht.min() + 1e-9)
    slope = float(np.polyfit(tn, hs, 1)[0])
    dur = max(ht.max(), max((b for _, b in B), default=ht.max()))   # union span so gesture_frac stays <= 1
    gfrac = sum(b - a for a, b in B) / dur if dur > 0 else np.nan
    rec.append(dict(trial=t, condition=r.condition, homing_slope=slope,
                    gesture_frac=gfrac, mean_sway=float(np.mean(S[1]))))
cm = pd.DataFrame(rec)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for cond, col in [("board", "#c44"), ("ground", "#48a")]:
    d = cm[cm.condition == cond]
    ax[0].scatter(d.gesture_frac, d.homing_slope, color=col, label=cond, alpha=.8)
    ax[1].scatter(d.mean_sway,   d.homing_slope, color=col, label=cond, alpha=.8)
ax[0].set_xlabel("fraction of trial gesturing"); ax[0].set_ylabel("semantic-homing slope")
ax[1].set_xlabel("mean board sway (deg/s)");    ax[1].set_ylabel("semantic-homing slope")
for a in ax: a.legend()
ax[0].set_title("Homing vs gesturing"); ax[1].set_title("Homing vs sway")
plt.tight_layout(); plt.show()
print(cm.groupby("condition")[["homing_slope", "gesture_frac", "mean_sway"]].mean().round(3))

## 5 · Bonus — does board sway *lead or lag* semantic homing?

The scatter in §4 asked whether homing and sway co-vary *per trial*. Here we ask a sharper,
**temporal** question: within a trial, when meaning approaches the target, does the body get
busier **before**, **during**, or **after**? We cross-correlate the two streams at a range of
lags and average the result across trials.

The mechanics:
1. Put both streams on a **common 10 Hz grid** (homing is only a few points per trial; the gyro
   is high-rate — we interpolate both).
2. **Linearly detrend + z-score** each — homing has a strong upward trend we don't want to
   dominate the correlation; we care about the *fluctuations* riding on it.
3. For each lag, correlate `homing(t)` with `sway(t − lag)`, so **positive lag = sway leads
   homing**. Average the lag curve across trials.

**Board trials only** (ground sway ≈ 0, so there's nothing to correlate). And the usual caveat,
harder here than anywhere: 2 dyads, ~14 s trials, ~5–10 homing points each → the lag axis is
coarse and this is strictly *hypothesis-generating*. Read the **shape and sign of the peak**, not
a p-value.

In [ ]:
# Cross-correlate homing vs sway at a range of lags, averaged over board trials.
# Reuses homing() and sway() from section 2. Positive lag = sway LEADS homing.
DT      = 0.1                                    # common-grid step (s) -> 10 Hz
MAX_LAG = 3.0                                    # inspect lags within +/- 3 s
lags    = np.arange(-MAX_LAG, MAX_LAG + DT / 2, DT)

def _detrend_z(y):
    """Remove a linear trend, then z-score. None if the series is flat."""
    x = np.arange(len(y))
    b, a = np.polyfit(x, y, 1)
    r = y - (a + b * x)
    sd = r.std()
    return r / sd if sd > 1e-9 else None

def xcorr_trial(trial_id, pair_id, trial_number):
    """Lag-correlation curve for one trial, aligned to `lags`. None if unusable."""
    H = homing(trial_id); S = sway(pair_id, trial_number)
    if H is None or S is None:
        return None
    ht, hs, _ = H; st, se = S
    t0, t1 = max(ht.min(), st.min()), min(ht.max(), st.max())
    if t1 - t0 < 2 * MAX_LAG + 1:               # need room for the lag window
        return None
    grid = np.arange(t0, t1, DT)
    h = _detrend_z(np.interp(grid, ht, hs))     # homing on the grid
    s = _detrend_z(np.interp(grid, st, se))     # sway envelope on the grid
    if h is None or s is None:
        return None
    n = len(grid); out = []
    for lag in lags:
        k = int(round(lag / DT))                # corr( h(t), s(t-lag) )
        a, b = (h[k:], s[:n - k]) if k >= 0 else (h[:n + k], s[-k:])
        out.append(float(np.corrcoef(a, b)[0, 1]) if len(a) >= 5 else np.nan)
    return np.array(out)

curves = []
for t in tx.trial_id.unique():
    r = tx[tx.trial_id == t].iloc[0]
    if r.condition != "board":                  # ground sway ~ 0 -> skip
        continue
    c = xcorr_trial(t, r.pair_id, r.trial_number)
    if c is not None:
        curves.append(c)

if not curves:
    print("No board trials with both streams long enough for a lagged cross-correlation.")
else:
    M = np.vstack(curves)
    mean, sem = np.nanmean(M, 0), np.nanstd(M, 0) / np.sqrt(np.sum(~np.isnan(M), 0))
    peak = int(np.nanargmax(np.abs(mean)))      # strongest coupling (either sign)

    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.axhline(0, color="#999", lw=.8); ax.axvline(0, color="#999", lw=.8, ls=":")
    ax.plot(lags, mean, "-", color="#c44", label=f"mean over {len(curves)} board trials")
    ax.fill_between(lags, mean - sem, mean + sem, color="#c44", alpha=.15)
    ax.plot(lags[peak], mean[peak], "o", color="#000")
    ax.set_xlabel("lag (s)   —   negative: sway trails homing   ·   positive: sway leads homing")
    ax.set_ylabel("cross-correlation"); ax.legend()
    ax.set_title("Do sway fluctuations lead or lag semantic homing? (board trials)")
    plt.tight_layout(); plt.show()

    lead = "sway LEADS homing" if lags[peak] > 0 else ("sway TRAILS homing" if lags[peak] < 0 else "synchronous")
    print(f"peak |correlation| = {mean[peak]:+.3f} at lag {lags[peak]:+.1f}s  ->  {lead}")
    print("Exploratory (2 dyads, short trials): read the shape/sign of the peak, not significance.")

## 6 · Bonus — are gesture onsets time-locked *just before* semantic-homing jumps?

§5 asked about the *board*; here we ask about the *hands*. When the clue-giver's cumulative clue
suddenly gets **closer** to the target (a semantic "jump"), is there a **gesture onset in the short
window just before** it?

The test is event-based:
1. A **homing jump** = a clue segment where similarity-to-target *rose* over the previous one.
2. A gesture **precedes** a jump if a gesture **onset** falls in the `W`-second window before it.
3. We compare the real *"fraction of jumps preceded by a gesture"* against a **time-shifted null**:
   the whole onset train is circularly shifted by a random offset, which keeps the *number* **and**
   the *clustering* of gestures intact and only breaks their alignment to the jumps. (A uniform-random
   scatter would be an unfair null — real gestures bunch inside speech, exactly where jumps live, so
   real would beat uniform even with no genuine timing relationship.)

**Read this honestly, not as proof of anticipation.** Because jumps are timestamped at speech-segment
*ends*, a gesture that merely overlaps the segment that produced the jump already counts as
"preceding" it. So this measures **gesture–speech co-occurrence with a slight lead**, not that the
hands lead meaning. And it's 2 dyads with sparse events → an exploratory *template*: read the
direction versus the null, never a p-value.

In [ ]:
# Event test: does a gesture ONSET fall just before a semantic-homing JUMP?
# Reuses homing() and gesture_bouts() from section 2.
W     = 2.0                                  # a gesture "precedes" a jump if it starts within W s before
NSHUF = 50                                   # circular-shift nulls per trial
rng   = np.random.default_rng(0)

def homing_jumps(trial_id):
    """Times of positive homing increments (the semantic 'jumps'), at clue-segment end times."""
    H = homing(trial_id)
    if H is None:
        return None
    ht, hs, _ = H
    if len(ht) < 3:
        return None
    j = ht[1:][np.diff(hs) > 0]             # segment times where similarity rose
    return j if len(j) else None

def preceded_fraction(jumps, onsets, W):
    """Fraction of jumps with >=1 gesture onset in (jump - W, jump]."""
    if len(jumps) == 0 or len(onsets) == 0:
        return np.nan
    onsets = np.asarray(onsets, dtype=float)
    return float(np.mean([np.any((onsets <= j) & (onsets > j - W)) for j in jumps]))

real, shuf, latency = [], [], []
for t in tx.trial_id.unique():
    J = homing_jumps(t); B = gesture_bouts(t)
    if J is None or not B:
        continue
    onsets = np.array([a for a, _ in B], dtype=float)
    dur = max(J.max(), onsets.max(), max(b for _, b in B))
    real.append(preceded_fraction(J, onsets, W))
    # Null: circularly shift the whole onset train by a random offset -> preserves the NUMBER and the
    # clustering/spacing of gestures, only breaks their alignment to the jumps (fairer than uniform scatter).
    sh = [preceded_fraction(J, np.sort((onsets + rng.uniform(0, dur)) % dur), W) for _ in range(NSHUF)]
    shuf.append(float(np.nanmean(sh)))
    for j in J:                             # latency from nearest preceding onset
        prev = onsets[onsets <= j]
        if len(prev):
            latency.append(j - prev.max())

real, shuf = np.array(real), np.array(shuf)
if np.isnan(real).all():
    print("Not enough trials with both gesture bouts and homing jumps to run this test.")
else:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    ax[0].bar(["gesture precedes\n(real)", "time-shifted\nnull"],
              [np.nanmean(real), np.nanmean(shuf)], color=["#48a", "#999"])
    ax[0].set_ylim(0, 1)
    ax[0].set_ylabel(f"fraction of jumps with a gesture\nonset within {W:.0f}s before")
    ax[0].set_title("Do gestures precede semantic jumps?")
    if latency:
        ax[1].hist(latency, bins=np.arange(0, 6.5, 0.5), color="#48a", alpha=.8)
        ax[1].axvline(W, color="#c44", ls="--", label=f"{W:.0f}s window")
        ax[1].set_xlabel("jump − nearest preceding gesture onset (s)")
        ax[1].set_ylabel("count"); ax[1].legend()
        ax[1].set_title("How long before a jump did the last gesture start?")
    plt.tight_layout(); plt.show()
    print(f"gesture-precedes-jump fraction:  real={np.nanmean(real):.2f}  "
          f"time-shifted null={np.nanmean(shuf):.2f}  (n={np.sum(~np.isnan(real))} trials)")
    print("Exploratory: jumps are at speech-segment ENDS, so this is gesture/speech CO-OCCURRENCE with a")
    print("slight lead, not proof gesture anticipates meaning. Read direction vs the null, not significance.")

## Recap & where this goes

- We aligned **meaning (language), motor (gesture), and body (sway)** on a single trial clock —
  a concrete multiscale, multimodal object.
- We reduced each trial to cross-modal summaries and looked for language↔body coupling,
  split by the **board/ground** manipulation.

**Caveats (state them):** 2 dyads, short trials, coarse gesture/sway summaries — everything is
*hypothesis-generating*. The value is the **template**: this is exactly the multiscale story a
group project would scale to the full Zenodo corpus, and the join keys (`pair_id` + `trial_number`,
trial-relative time) are the same throughout.

**Ideas to extend:** cross-correlate the homing curve with the sway envelope at a lag;
test whether gesture *onsets* precede jumps in semantic similarity; bring in the prosody
streams in `TS_acoustics/` (envelope, f0) as a fourth channel.